# Pendulum

Combines `pendulum.xml` (a swinging double-jointed arm tethered by a tendon to a free-floating cylinder) with the interactive viewer logic from `viewer.py`.

**macOS note:** `mujoco.viewer.launch_passive` must run on the main thread of a process started via `mjpython` (MuJoCo's bundled launcher), not a regular Python/Jupyter kernel. If the interactive viewer cell below doesn't open a window or raises a threading error, that's why — run the equivalent code as a standalone script instead:

```bash
.venv/bin/mjpython viewer.py
```

The last cell renders offscreen frames instead, which works reliably from any kernel.

In [ ]:
import mujoco

model = mujoco.MjModel.from_xml_path("pendulum.xml")
data = mujoco.MjData(model)

print(f"Bodies: {model.nbody}, Joints: {model.njnt}, qpos DOFs: {model.nq}")

### Interactive viewer (requires `mjpython` on macOS)

In [ ]:
import time
import mujoco.viewer

with mujoco.viewer.launch_passive(model, data) as viewer:
  # Close the viewer automatically after 30 wall-seconds.
  start = time.time()
  while viewer.is_running() and time.time() - start < 30:
    step_start = time.time()

    # mj_step can be replaced with code that also evaluates
    # a policy and applies a control signal before stepping the physics.
    mujoco.mj_step(model, data)

    # Example modification of a viewer option: toggle contact points every two seconds.
    with viewer.lock():
      viewer.opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = int(data.time % 2)

    # Pick up changes to the physics state, apply perturbations, update options from GUI.
    viewer.sync()

    # Rudimentary time keeping, will drift relative to wall clock.
    time_until_next_step = model.opt.timestep - (time.time() - step_start)
    if time_until_next_step > 0:
      time.sleep(time_until_next_step)

### In-notebook visualization (offscreen render, works regardless of `mjpython`)

In [ ]:
import matplotlib.pyplot as plt

data2 = mujoco.MjData(model)  # fresh state, independent of the viewer cell above
renderer = mujoco.Renderer(model, height=300, width=400)

n_frames = 6
duration = 2.0  # seconds
steps_per_frame = int((duration / model.opt.timestep) / n_frames)

frames = []
for _ in range(n_frames):
    for _ in range(steps_per_frame):
        mujoco.mj_step(model, data2)
    renderer.update_scene(data2)
    frames.append(renderer.render())

fig, axes = plt.subplots(1, n_frames, figsize=(3 * n_frames, 3))
for ax, frame, i in zip(axes, frames, range(n_frames)):
    ax.imshow(frame)
    ax.set_title(f"t={(i + 1) * steps_per_frame * model.opt.timestep:.2f}s")
    ax.axis("off")
plt.tight_layout()
plt.show()

renderer.close()